In [1]:
import os
os.environ["OPENAI_AGENTS_DISABLE_TRACING"] = "1"

In [2]:

from openai import AsyncOpenAI
from agents import Agent, Runner, OpenAIChatCompletionsModel

In [3]:
local_model = OpenAIChatCompletionsModel(
    model = "minimax-m2:cloud",
    openai_client=AsyncOpenAI(
        api_key="ollama",
        base_url= "http://localhost:11434/v1"
    )
)

In [4]:
agent = Agent(
    name="Greeter",                                    
    instructions="You are a friendly assistant. Be concise.", 
    model=local_model 
)

# # Step 2: Run it (sync version — good for scripts)
result = await Runner.run(agent, "Hello! What is capital of America?")

# # Step 3: Get the output
print(result.final_output)

The capital of the United States is **Washington, D.C.** (Washington, District of Columbia).


In [5]:
async def main():
    result = await Runner.run(agent, "Why is Python popular for AI?")
    
    print("=" * 50)
    print(f"Final Output:  {result.final_output[:100]}...")
    print(f"Last Agent:    {result.last_agent.name}")
    print(f"New Items:     {len(result.new_items)}")
    print(f"Raw Responses: {len(result.raw_responses)}")
    print("=" * 50)

# Execute
await main()

Final Output:  - **Simple, readable syntax** – lets researchers and engineers focus on algorithms rather than boile...
Last Agent:    Greeter
New Items:     1
Raw Responses: 1


In [6]:
result = await Runner.run(agent, "What is a list in Python?")
print(f"Turn 1: {result.final_output}\n")

Turn 1: A **list** in Python is an ordered, mutable collection of items that can hold objects of different types.

**Key characteristics:**
- Ordered (maintains insertion order)
- Mutable (can be modified in place)
- Allows duplicate values
- Indexed (starts at 0)

**Creating a list:**
```python
my_list = [1, 2, 3, "hello", True]
```

**Common operations:**
```python
my_list.append(4)        # Add to end
my_list.insert(0, 0)     # Insert at index
my_list.remove("hello")  # Remove by value
popped = my_list.pop()   # Remove and return last item
my_list[0]               # Access by index (returns 0)
my_list[1:3]             # Slice (returns [2, 3])
```

Lists are one of the most frequently used data structures in Python.



In [7]:
new_input = result.to_input_list() + [{"role": "user", "content": "How do I sort one?"}]
result = await Runner.run(agent, new_input)
print(f"Turn 2: {result.final_output}\n")

Turn 2: Use the `sort()` method to sort in place, or `sorted()` to return a new list:

```python
# In place (modifies original list)
numbers = [3, 1, 4, 1, 5]
numbers.sort()
# numbers is now [1, 1, 3, 4, 5]

# Returns new list (original unchanged)
numbers = [3, 1, 4, 1, 5]
sorted_numbers = sorted(numbers)
# sorted_numbers = [1, 1, 3, 4, 5]
```

**Reverse sorting:**
```python
numbers.sort(reverse=True)        # In place
sorted(numbers, reverse=True)     # New list
```

**Sort with custom key:**
```python
words = ["banana", "apple", "cherry"]
words.sort(key=len)  # Sort by length: ["apple", "banana", "cherry"]
```



In [8]:
new_input = result.to_input_list() + [{"role": "user", "content": "What about reverse sort?"}]
result = await Runner.run(agent, new_input)
print(f"Turn 3: {result.final_output}")

Turn 3: Two ways to reverse sort:

```python
numbers = [3, 1, 4, 1, 5]

# Method 1: sort() with reverse=True
numbers.sort(reverse=True)

# Method 2: sorted() with reverse=True
numbers = [3, 1, 4, 1, 5]
result = sorted(numbers, reverse=True)
```

**Also note:** You can reverse a list without sorting using `[::-1]` or `.reverse()`:

```python
numbers = [1, 2, 3]
numbers[::-1]        # Returns [3, 2, 1] (new list)
numbers.reverse()    # Modifies in place to [3, 2, 1]
```


In [16]:
agent = Agent(
    name="greeter", 
    instructions="Explain briefly.",
    model=local_model  # This was missing!
)

# Streaming — see tokens as they arrive
result = Runner.run_streamed(agent, "What is machine learning in 2 sentences?")

async for event in result.stream_events():
    if event.type == "raw_response_event" and hasattr(event.data, 'delta'):
        print(event.data.delta, end="", flush=True)

print(f"\n\n--- Final: {result.final_output}")

The user wants a brief explanation of machine learning in just 2 sentences. I need to condense the concept into two clear, concise sentences that capture the essence of what machine learning is.Machine learning is a type of artificial intelligence that allows computers to learn from data and improve their performance on tasks without being explicitly programmed. It involves algorithms that identify patterns in data and use those patterns to make predictions or decisions.

--- Final: Machine learning is a type of artificial intelligence that allows computers to learn from data and improve their performance on tasks without being explicitly programmed. It involves algorithms that identify patterns in data and use those patterns to make predictions or decisions.
